# Faruq-v3 LFDet AFAB breadth screening

Seed-42 discovery for **AF1** (adaptive patch high-pass), **AF2** (entropy-conditioned directional amplitude suppression), and **AF12** (both). Patch size 32, r=0.05, gamma=0.1 follow LFDet. Overlap 0.50 is a paper-tested setting deliberately frozen for breadth-search tractability; any retained arm must later confirm overlap 0.75. AFAB runs at both training and inference. Test is never extracted/opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os,shutil,subprocess,sys,tarfile,time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/lfdet-afab-frequency-input-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
cmd=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    r=subprocess.run(cmd)
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('clone gagal')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact,resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan GPU'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt','experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json'))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as a: a.extractall('/content',filter='data')
GROUPED=DATA_ROOT/'faruq_grouped_summary.json'; assert GROUPED.is_file(); assert not (DATA_ROOT/'test').exists()
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-lfdet-afab-screening-v1'
print(torch.cuda.get_device_name(0),OUTPUT)


In [ ]:
cmd=[sys.executable,'-m','pytest','-q','tests/test_afab.py']
print('STATIC CHECK:',' '.join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_afab_screening','--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),'--control-summary',str(CONTROL),'--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('MENJALANKAN:',' '.join(cmd),flush=True)
p=subprocess.run(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT); print(p.stdout)
if p.returncode: raise RuntimeError(f'AFAB screen gagal {p.returncode}')


In [ ]:
import json,pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/lfdet_afab_seed42_screening.json'; r=json.loads(SUMMARY.read_text())
assert r['test_opened'] is False and r['test_images_accessed'] is False
rows=[{'model':k,**v} for k,v in r['controls'].items()]+[{'model':k,**v} for k,v in r['candidate'].items()]
display(pd.DataFrame(rows).style.format({'macro_map50_95':'{:.2%}','bottom3_class_map50_95':'{:.2%}','worst_class_map50_95':'{:.2%}'}))
for arm in ('AF1','AF2','AF12'):
    print(arm,r['decisions'][arm]['decision'],r['decisions'][arm]['delta_vs_D0FT'])
print('TRANSFER CHOICES:',r['transfer_choices']); print('SUMMARY:',SUMMARY)
